# Momentum Transformer: Colab Training
This notebook contains the complete pipeline (data downloading, feature engineering, model definition, and training) in a single file, making it easy to run on Google Colab with a GPU.

## 1. Install Dependencies
Run this cell to install required packages.

In [ ]:
!pip install yfinance pandas numpy torch tqdm scipy

## 2. All-in-One Source Code
We consolidate the modular codebase into this single cell for Colab compatibility.

In [ ]:
# ========================================
# data.py
# ========================================

import yfinance as yf
import pandas as pd
import os

def download_data(tickers, start_date, end_date, output_path=None):
    """
    Downloads OHLCV data for a list of tickers.
    """
    print(f"Downloading data for {tickers} from {start_date} to {end_date}...")
    data = yf.download(tickers, start=start_date, end=end_date)
    
    # yfinance returns a MultiIndex column dataframe if multiple tickers are passed
    # We want to stack it to have 'Ticker' as a level or melt it
    if len(tickers) > 1:
        data = data.stack(level=1).rename_axis(['Date', 'Ticker']).reset_index()
    else:
        data = data.reset_index()
        data['Ticker'] = tickers[0]
        
    if output_path:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        data.to_csv(output_path, index=False)
        print(f"Data saved to {output_path}")
        
    return data


# ========================================
# features.py
# ========================================

import pandas as pd
import numpy as np
import os

def compute_macd(close, short_window, long_window):
    """Compute MACD signal for a given short and long window."""
    ewm_short = close.ewm(span=short_window, adjust=False).mean()
    ewm_long = close.ewm(span=long_window, adjust=False).mean()
    return ewm_short - ewm_long

def compute_features(df):
    """
    Computes momentum features for each ticker.
    df is expected to have ['Date', 'Ticker', 'Close', 'Open', 'High', 'Low', 'Volume']
    """
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values(by=['Ticker', 'Date']).reset_index(drop=True)
    
    processed_dfs = []
    
    for ticker, group in df.groupby('Ticker'):
        group = group.copy()
        
        # Daily return
        group['Return_1d'] = group['Close'].pct_change()
        
        # Ex-ante volatility: EWMA of squared returns, then sqrt (Paper Section VI.A)
        # sigma^2_t = lambda * sigma^2_{t-1} + (1-lambda) * r^2_{t-1}, lambda = 1 - 1/60
        lam = 1.0 - 1.0 / 60.0
        sq_rets = group['Return_1d'] ** 2
        ewma_var = sq_rets.ewm(alpha=1 - lam, adjust=False).mean()
        group['Vol_60d'] = np.sqrt(ewma_var)
        
        # Forward fill and drop leading NaNs for Vol
        group['Vol_60d'] = group['Vol_60d'].ffill()
        
        # Multi-horizon returns (1, 21, 63, 126, 252 days)
        horizons = [1, 21, 63, 126, 252]
        for h in horizons:
            # We use log returns over horizon to avoid compounding issues, or simple returns
            group[f'Return_{h}d'] = group['Close'].pct_change(periods=h)
            
            # Volatility scaling
            # Paper uses returns normalized by ex-ante volatility
            group[f'Return_{h}d_scaled'] = group[f'Return_{h}d'] / (group['Vol_60d'] * np.sqrt(h) + 1e-8)
            
        # MACD indicators (8, 24), (16, 28), (32, 96)
        # Paper Eq.4: m̄_{t,s,l} = (v_{t,s} - v_{t,l}) / (sigma_t * sqrt(l))
        # where v_{t,s/l} is the EMA of Close, sigma_t is daily vol
        macd_pairs = [(8, 24), (16, 28), (32, 96)]
        for s, l in macd_pairs:
            group[f'MACD_{s}_{l}'] = compute_macd(group['Close'], s, l)
            # Correct normalisation: EMA difference is in price units, divide by (sigma * Close * sqrt(l))
            # to get a dimensionless signal (volatility-normalised price difference)
            group[f'MACD_{s}_{l}_scaled'] = group[f'MACD_{s}_{l}'] / (group['Close'] * group['Vol_60d'] * np.sqrt(l) + 1e-8)
            
        # Target: next day return
        # Paper uses continuous sizing z_t in (-1, 1). We train to maximize Sharpe.
        # But we need the next day's raw return to calculate the Sharpe loss.
        group['Target_Return'] = group['Return_1d'].shift(-1)
        
        processed_dfs.append(group)
        
    final_df = pd.concat(processed_dfs, ignore_index=True)
    # Drop rows where we don't have enough history for the 252d return or vol
    final_df = final_df.dropna().reset_index(drop=True)
    return final_df


# ========================================
# dataset.py
# ========================================

import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

class MomentumDataset(Dataset):
    def __init__(self, df, seq_length=252, step=1):
        self.seq_length = seq_length
        self.step = step
        self.features = [
            'Return_1d_scaled', 'Return_21d_scaled', 'Return_63d_scaled', 
            'Return_126d_scaled', 'Return_252d_scaled',
            'MACD_8_24_scaled', 'MACD_16_28_scaled', 'MACD_32_96_scaled'
        ]
        
        self.indices = []
        self.data_X = []
        self.data_y = []
        self.data_vol = []
        self.data_date = []
        self.data_ticker = []
        
        # Group by ticker to prevent sequence overlap between different assets
        for ticker, group in df.groupby('Ticker'):
            group = group.sort_values('Date').reset_index(drop=True)
            self.data_X.append(group[self.features].values.astype(np.float32))
            self.data_y.append(group['Target_Return'].values.astype(np.float32))
            self.data_vol.append(group['Vol_60d'].values.astype(np.float32))
            self.data_date.append(group['Date'].astype(str).values)
            self.data_ticker.append(ticker)
            
            ticker_idx = len(self.data_ticker) - 1
            for i in range(0, len(group) - seq_length + 1, self.step):
                self.indices.append((ticker_idx, i))
                
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        t_idx, start_idx = self.indices[idx]
        return {
            'X': torch.tensor(self.data_X[t_idx][start_idx : start_idx + self.seq_length]),
            'y': torch.tensor(self.data_y[t_idx][start_idx : start_idx + self.seq_length]),
            'vol': torch.tensor(self.data_vol[t_idx][start_idx : start_idx + self.seq_length]),
            'date': self.data_date[t_idx][start_idx + self.seq_length - 1],
            'ticker': self.data_ticker[t_idx]
        }

def get_dataloaders(df, seq_length=252, batch_size=64, train_end='2018-01-01', val_end='2020-01-01'):
    df['Date'] = pd.to_datetime(df['Date'])
    
    train_df = df[df['Date'] < train_end]
    val_df = df[(df['Date'] >= train_end) & (df['Date'] < val_end)]
    test_df = df[df['Date'] >= val_end]
    
    train_dataset = MomentumDataset(train_df, seq_length, step=seq_length)
    val_dataset = MomentumDataset(val_df, seq_length, step=1)
    test_dataset = MomentumDataset(test_df, seq_length, step=1)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    # Val and Test shouldn't be shuffled for evaluation ordering if needed, but for Sharpe it doesn't matter much.
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader, test_loader


# ========================================
# model.py
# ========================================

import torch
import torch.nn as nn
import torch.nn.functional as F

class GLU(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_model)
        self.fc2 = nn.Linear(d_model, d_model)
        
    def forward(self, x):
        return self.fc1(x) * torch.sigmoid(self.fc2(x))

class GRN(nn.Module):
    """Gated Residual Network"""
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_model)
        self.elu = nn.ELU()
        self.fc2 = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.gate = GLU(d_model)
        self.norm = nn.LayerNorm(d_model)
        
    def forward(self, x):
        res = x
        x = self.fc1(x)
        x = self.elu(x)
        x = self.fc2(x)
        x = self.dropout(x)
        x = self.gate(x)
        return self.norm(res + x)

class VariableSelectionNetwork(nn.Module):
    def __init__(self, num_features, d_model, dropout=0.1):
        super().__init__()
        self.num_features = num_features
        # We assume input is (batch, seq, num_features). We embed each feature separately.
        # Since our inputs are continuous, we can just use a linear layer for each.
        self.feature_embeddings = nn.ModuleList([
            nn.Linear(1, d_model) for _ in range(num_features)
        ])
        
        self.flattened_grn = GRN(num_features * d_model, dropout)
        self.softmax_linear = nn.Linear(num_features * d_model, num_features)
        
        self.single_feature_grns = nn.ModuleList([
            GRN(d_model, dropout) for _ in range(num_features)
        ])
        
    def forward(self, x):
        # x shape: (batch, seq, num_features)
        batch_size, seq_len, _ = x.shape
        
        # Embed each feature: output list of (batch, seq, d_model)
        embedded_features = []
        for i in range(self.num_features):
            feat = x[:, :, i].unsqueeze(-1) # (batch, seq, 1)
            embedded = self.feature_embeddings[i](feat)
            embedded_features.append(embedded)
            
        # Concatenate all embedded features
        # (batch, seq, num_features * d_model)
        concat_features = torch.cat(embedded_features, dim=-1)
        
        # Compute variable selection weights
        weight_network = self.flattened_grn(concat_features)
        weights = torch.softmax(self.softmax_linear(weight_network), dim=-1) # (batch, seq, num_features)
        
        # Process each feature through its own GRN and apply weights
        processed_features = []
        for i in range(self.num_features):
            processed = self.single_feature_grns[i](embedded_features[i]) # (batch, seq, d_model)
            weight = weights[:, :, i].unsqueeze(-1) # (batch, seq, 1)
            processed_features.append(processed * weight)
            
        # Sum over features
        # (batch, seq, d_model)
        output = torch.stack(processed_features, dim=-1).sum(dim=-1)
        return output, weights

class InterpretableMultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_model = d_model
        self.d_k = d_model // n_heads
        
        self.q_layers = nn.Linear(d_model, d_model)
        self.k_layers = nn.Linear(d_model, d_model)
        # Value weights are shared across heads in TFT (Interpretable MHA)
        self.v_layer = nn.Linear(d_model, self.d_k)
        
        # W_H: output projection from d_k -> d_model (TFT paper requires this)
        self.W_H = nn.Linear(self.d_k, d_model)
        
        self.attention = None
        
    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        
        # Linear projections for Q and K (per-head)
        q = self.q_layers(q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        k = self.k_layers(k).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        # V is SHARED across heads: single projection (batch, seq, d_k)
        v_shared = self.v_layer(v) # (batch, seq, d_k)
        
        # Attention scores (per head)
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.d_k ** 0.5)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
            
        attn = torch.softmax(scores, dim=-1) # (batch, n_heads, seq, seq)
        self.attention = attn # Save for interpretability
        
        # Apply each head's attention to the SHARED V
        # v_shared: (batch, seq, d_k) -> (batch, 1, seq, d_k) for broadcast
        v_expanded = v_shared.unsqueeze(1) # (batch, 1, seq, d_k)
        context_per_head = torch.matmul(attn, v_expanded) # (batch, n_heads, seq, d_k)
        
        # Average across heads (Interpretable MHA: average instead of concatenate)
        context = context_per_head.mean(dim=1) # (batch, seq, d_k)
        
        # W_H output projection
        output = self.W_H(context) # (batch, seq, d_model)
        return output

class MomentumTransformer(nn.Module):
    def __init__(self, num_features=8, d_model=64, n_heads=4, dropout=0.4):
        super().__init__()
        
        self.vsn = VariableSelectionNetwork(num_features, d_model, dropout)
        
        # LSTM encoder for local processing
        self.lstm = nn.LSTM(
            input_size=d_model, 
            hidden_size=d_model, 
            batch_first=True
        )
        
        self.lstm_norm = nn.LayerNorm(d_model)
        self.lstm_gate = GLU(d_model)
        
        # Interpretable Multi-Head Attention
        self.attention = InterpretableMultiHeadAttention(d_model, n_heads)
        self.attn_norm = nn.LayerNorm(d_model)
        self.attn_gate = GLU(d_model)
        
        # Feed-forward
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ELU(),
            nn.Linear(d_model, d_model)
        )
        self.ffn_norm = nn.LayerNorm(d_model)
        self.ffn_gate = GLU(d_model)
        
        # Final output layer predicting position size z_t
        self.output_layer = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ELU(),
            nn.Linear(d_model // 2, 1),
            nn.Tanh()
        )
        
    def forward(self, x):
        # x: (batch, seq, num_features)
        
        # 1. Variable Selection Network
        vsn_out, var_weights = self.vsn(x) # (batch, seq, d_model)
        
        # 2. LSTM
        lstm_out, _ = self.lstm(vsn_out) # (batch, seq, d_model)
        # Add & Norm with Gate
        lstm_out = self.lstm_norm(vsn_out + self.lstm_gate(lstm_out))
        
        # 3. Attention (Masked to prevent lookahead)
        seq_len = x.size(1)
        mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0).to(x.device)
        attn_out = self.attention(lstm_out, lstm_out, lstm_out, mask)
        # Add & Norm with Gate
        attn_out = self.attn_norm(lstm_out + self.attn_gate(attn_out))
        
        # 4. FFN
        ffn_out = self.ffn(attn_out)
        out = self.ffn_norm(attn_out + self.ffn_gate(ffn_out))
        
        positions = self.output_layer(out) # (batch, seq, 1)
        z = positions.squeeze(-1) # (batch, seq)
        
        return z, var_weights, self.attention.attention


# ========================================
# train.py
# ========================================

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

def sharpe_loss(returns, ann_factor=252):
    """
    Computes negative annualized Sharpe ratio as the loss.
    Matches the paper's definition: sqrt(252) * E[R] / std(R).
    Uses population std (ddof=0) to match the expectation formula exactly.
    returns: tensor of daily portfolio returns over Omega.
    """
    mean_ret = torch.mean(returns)
    # Population std (ddof=0) matches paper's Var[] definition
    std_ret = torch.std(returns, unbiased=False)
    
    # Add a small epsilon to avoid division by zero
    sharpe = (mean_ret * np.sqrt(ann_factor)) / (std_ret + 1e-8)
    return -sharpe

def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    epoch_returns = []
    
    # 3bps average transaction cost
    cost_bps = 0.0003
    tgt_vol_daily = 0.15 / np.sqrt(252)
    
    for batch in tqdm(loader, desc="Training"):
        X = batch['X'].to(device) # (batch, seq, features)
        y = batch['y'].to(device) # (batch, seq)
        vol = batch['vol'].to(device) # (batch, seq)
        
        optimizer.zero_grad()
        z, _, _ = model(X) # (batch, seq)
        
        scaled_z = z * (tgt_vol_daily / (vol + 1e-8))
        
        # Calculate turnover across the sequence
        prev_scaled_z = torch.cat([torch.zeros_like(scaled_z[:, :1]), scaled_z[:, :-1]], dim=1)
        turnover = torch.abs(scaled_z - prev_scaled_z)
        
        # Raw strategy returns minus transaction cost penalty (Eq 12)
        strat_returns = scaled_z * y - cost_bps * turnover
        
        # Flatten into Omega tensor containing all asset-time pairs (batch * seq)
        omega_returns = strat_returns.reshape(-1)
        
        # Compute Sharpe over entire Omega tensor (Eq 11)
        loss = sharpe_loss(omega_returns)
        loss.backward()
        
        # Clip gradients (Paper uses 100 for TFT)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=100.0)
        optimizer.step()
        
        total_loss += loss.item()
        epoch_returns.append(omega_returns.detach())
        
    epoch_returns = torch.cat(epoch_returns)
    epoch_sharpe = -sharpe_loss(epoch_returns).item()
    
    return total_loss / len(loader), epoch_sharpe

def eval_epoch(model, loader, device):
    model.eval()
    all_returns = []
    
    cost_bps = 0.0003
    tgt_vol_daily = 0.15 / np.sqrt(252)
    
    with torch.no_grad():
        for batch in loader:
            X = batch['X'].to(device)
            y = batch['y'].to(device)
            vol = batch['vol'].to(device)
            
            z, _, _ = model(X)
            
            scaled_z = z * (tgt_vol_daily / (vol + 1e-8))
            prev_scaled_z = torch.cat([torch.zeros_like(scaled_z[:, :1]), scaled_z[:, :-1]], dim=1)
            turnover = torch.abs(scaled_z - prev_scaled_z)
            
            strat_returns = scaled_z * y - cost_bps * turnover
            all_returns.append(strat_returns.reshape(-1))
            
    all_returns = torch.cat(all_returns)
    sharpe = -sharpe_loss(all_returns).item()
    return sharpe

def train_model():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    df = pd.read_csv("data/features.csv")
    # Paper uses batch size 64 for TFT
    train_loader, val_loader, test_loader = get_dataloaders(df, batch_size=64)
    
    model = MomentumTransformer(num_features=8, d_model=64, n_heads=4).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
    
    best_val_sharpe = -float('inf')
    patience = 25
    patience_counter = 0
    epochs = 300
    
    os.makedirs("models", exist_ok=True)
    
    for epoch in range(epochs):
        train_loss, train_sharpe = train_epoch(model, train_loader, optimizer, device)
        val_sharpe = eval_epoch(model, val_loader, device)
        
        scheduler.step(val_sharpe)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {train_loss:.4f} | Train Sharpe: {train_sharpe:.4f} | Val Sharpe: {val_sharpe:.4f}")
        
        if val_sharpe > best_val_sharpe:
            best_val_sharpe = val_sharpe
            patience_counter = 0
            torch.save(model.state_dict(), "models/best_model.pth")
            print("  --> Saved new best model!")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs.")
                break
                
    print("Training complete.")
    



## 3. Execute Pipeline
Run the following cell to download data, compute features, and start training.

In [ ]:

import os

# 1. Download Data
tickers = [
    'SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 
    'TLT', 'IEF', 'SHY', 'LQD', 'HYG', 
    'GLD', 'SLV', 'USO', 'UNG', 'DBA', 
    'VNQ', 'XLF', 'XLK', 'XLV', 'XLE',
    'AAPL', 'MSFT', 'AMZN', 'GOOGL', 'META', 'NVDA', 'TSLA', 'BRK-B', 'UNH', 'JNJ',
    'JPM', 'V', 'PG', 'MA', 'HD', 'CVX', 'MRK', 'ABBV', 'LLY', 'PEP',
    'KO', 'BAC', 'AVGO', 'TMO', 'COST', 'MCD', 'CSCO', 'WMT', 'CRM', 'DIS',
    'PFE', 'DHR', 'NKE', 'XOM', 'TXN', 'UNP', 'LIN', 'ADBE', 'VZ', 'PM',
    'NEE', 'RTX', 'QCOM', 'INTC', 'AMD', 'HON', 'UPS', 'AMGN', 'IBM', 'BA'
]
df_raw = download_data(tickers, "2005-01-01", "2024-01-01", "data/raw_data.csv")
print("Data downloaded!")

# 2. Compute Features
df_features = compute_features(df_raw)
os.makedirs("data", exist_ok=True)
df_features.to_csv("data/features.csv", index=False)
print("Features computed!")

# 3. Train Model
# The train_model() function defined above will automatically load data/features.csv, 
# instantiate the model, and train using PyTorch on GPU if available.
train_model()
